In [1]:
%load_ext autoreload
%autoreload 2

# Labelled examples

This exploratory notebook serves for the creation of labelled examples

In [3]:
import pandas as pd
import json
from collections import Counter
import numpy as np
import pandas as pd
import seaborn as sns
import spacy
import re
import pycountry
from src.text_processing_functions import *
from src.LLM_functions import *
from src.plot_functions import *
from src.data import *
from src.hazard_def import *
from src.impact_def import *
import copy as cp
from random import randrange, randint

In [7]:
###### FILE PATHS
json_file = DATA_IN_JSONS+'filtered_report_types_nat_hazards_summary-header.json'

##### Open and read the JSON file
with open(json_file, 'r') as json_file:
    filtered_reports = json.load(json_file)

#### Already labelled reports
reports_labelled = pd.read_csv(DATA_LABELLED+"labelled_example.csv", encoding='utf-8')
save_name = 'labelled_example_haz-subtype-emdat_laura.csv'

In [8]:
# Randomly select 12 reports
# random_report = [randint(0, len(filtered_reports)) for i in range(1)]
# random_appealCode = [filtered_reports[i]['appealCode'] for i in random_report]

#Fix the selection :
random_appealCode = ['MDRLA009',
 'MDRMG020',
 'MDRNI012',
 'MDRBZ006',
 'MDRCN006',
 'MDRBD022',
 'MDRYE011',
 'MDRS2001',
 'MDRIQ014',
 'MDRGN015',
 'MDRSV012',
 'MDRMY003',
 'MDRBD015']

## Report labelling

Choose a hazard directory. In the case below we use : \
hazard_all_subtype_emdat = {
“drought”, 
“forest fire”, “land fire”, 
“ground movement”, “tsunami”, 
“avalanche”, “landslide”, “rockfall”, “sudden subsidence”, “mudslide", 
“ash fall”, “lava flow”, “pyroclastic flow”, “lahar”, 
“coastal flood”, “flash flood”, “riverine flood”, “ice jam flood”,
“rogue wave”, “seiche”, 
”coldwave”, “heatwave”, “severe winter conditions”, 
“derecho”, “hail”, “lightning/thunderstorm”, “sand/dust storm”,  “winter storm/blizzard”, “storm surge”, “tornado”, “extra-tropical storm”, “tropical cyclone”
}

The flood subtype being hard to differentiate, we will assign hazard to "flash flood" when the text mention heavy rain, "riverine flood" if nothing specific is mentioned. 


Choose a hazard dict : 
hazard_subtype_emdat = {
'Drought': r"drought.", 

'Wildfire': r"wildfire.|forest fire.|land fire." , 

‘Earthquake’ : r”ground movement.|tsunami.”, 

‘Mass movement’: r"avalanche.|landslide.|rockfall.|sudden subsidence.|mudslide.",

‘Volcanic activity’ : r“ash fall.|lava flow.|pyroclastic flow.|lahar”, 

'Flood': r"(coastal flood.|flash flood.|riverine flood.|ice jam flood.)",

‘Wave action’ : r“rogue wave.|seiche”,

‘Extreme temperature’ : r”coldwave.|heatwave.|severe winter conditions.”, 

‘Storm’ : r”derecho.|hail.|lightning.|winterstorm.|storm surge.|tornado.|winter storm.|extra-tropical storm.|tropical storm.”
}


In [9]:
def add_new_labelled_report(df_combined_labelled, df_rew_rep) :
    new_appealCode = df_rew_rep['appealCode'][0]
    if new_appealCode in df_combined_labelled.appealCode.unique() :
        #Remove the old version
        df_combined_labelled = df_combined_labelled.loc[df_combined_labelled.appealCode != new_appealCode]
        #Add the new df
        df_combined_labelled = pd.concat([df_combined_labelled, df_rew_rep], axis=0)
    else :
        df_combined_labelled = pd.concat([df_combined_labelled, df_rew_rep], axis=0)
    return df_combined_labelled

In [11]:
appealCode_cecily = ['MDRAF014', 'MDRCM036', 'MDRDZ008', 'MDRSY006', 'MDRBR010', 'MDRID013', 'MDR55001']
appealCode_to_label = reports_labelled.appealCode.unique().tolist() + appealCode_cecily# + random_appealCode[:2]
df_combined_labelled = pd.read_csv(DATA_LABELLED+'labelled_example_haz-subtype-emdat_14-02.csv')
appealCode_missing = [i for i in appealCode_to_label if i not in df_combined_labelled.appealCode.unique()]
# appealCode_to_label[~appealCode_to_label.isin(df_combined_labelled.appealCode.unique())]

In [13]:
## Create an initial empty directory 
dict_ = [
    {"hazardType": None,
     "hazardSubtypes" : None,
     "country" : None,
     "region" : None,
     "city" : None,
     "locationAnnotation" : None,
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    }]
df_combined_labelled = pd.DataFrame(dict_)
df_combined_labelled['appealCode'] = None
df_combined_labelled['reportDate'] = None
df_combined_labelled['Country'] = None

In [9]:
i = 0
appealCode_i = appealCode_missing[i]
found = False
for report in filtered_reports :
    if report['appealCode']==appealCode_i :
        print(report['date'])
        if not found :
            for sent in report['nathaz_text'] :
                print(sent)
            found = True
        # for sent in report['nathaz_text'] :
        #     print(sent)
if appealCode_i in reports_labelled.appealCode.unique() :
    print(reports_labelled.loc[reports_labelled.appealCode==appealCode_i])

24/05/2024
30
1 DREF Operation Nº MDRBR010 Glide N° FL-2021-000204-BRA Operation start date: 16 December 2021 Timeframe: 5 months (2-month extension) Operation end date: 31 May 2022 Final Report publication: 24 May 2024 Reporting period covered by this update: 16 December 2021 – 31 May 2022 IFRC Category allocated to the of the disaster or crisis: Yellow DREF allocated: 342,866 Swiss francs (an increase from the original CHF 261,223) Total number of people affected: Over 1 million in both states Number of people to be assisted: 4,000 (800 families) States affected: Bahia, Minas Gerais Areas targeted: Bahía (Jucuruçu and Medeiros Neto) Minas Gerais Host National Society presence: The Brazilian Red Cross (BRC) has its national headquarters in Rio de Janeiro and 21 branches with 6,000 volunteers and 300 staff members.
Red Cross Red Crescent Movement partners actively involved in the operation: International Federation of the Red Cross (IFRC) and International Committee of the Red Cross (I

In [15]:
dict_=[
    {"hazardType": "Storm",
     "hazardSubtypes" : "tropical storm",
     "country" : "Vanuatu",
     "region" : None,
     "city" : None,
     "locationAnnotation" : ["The Vanuatu Red Cross Society (VRCS) has mobilized over 200 volunteers and over 30 staff, working with the humanitarian effort to run International Appeal operations update Pacific: Tropical Cyclone Pam Kiribati Red Cross Society built a “tippy-tap” on Tamana Island, Kiribati, to promote effective hand washing and has encouraged the local community to construct this useful tool near their homes."
                            ],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Pam",
    },
    {"hazardType": "Storm",
     "hazardSubtypes" : "tropical storm",
     "country" : "Kiribati",
     "region" : ["Tamana", "Arorae", "Onotoa", "Nonouti"],
     "city" : None,
     "locationAnnotation" : ["The Vanuatu Red Cross Society (VRCS) has mobilized over 200 volunteers and over 30 staff, working with the humanitarian effort to run International Appeal operations update Pacific: Tropical Cyclone Pam Kiribati Red Cross Society built a “tippy-tap” on Tamana Island, Kiribati, to promote effective hand washing and has encouraged the local community to construct this useful tool near their homes.",
                             "The four outer atolls of Tamana, Arorae, Onotoa and Nonouti in Southern Kiribati were struck by strong winds, causing extensive damage to houses, and inundations from storm surges."
                            ],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Pam",
    },
    {"hazardType": "Storm",
     "hazardSubtypes" : "storm surge",
     "country" : "Kiribati",
     "region" : ["Tamana", "Arorae", "Onotoa", "Nonouti"],
     "city" : None,
     "locationAnnotation" : ["The four outer atolls of Tamana, Arorae, Onotoa and Nonouti in Southern Kiribati were struck by strong winds, causing extensive damage to houses, and inundations from storm surges."
                            ],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Flood",
     "hazardSubtypes" : "coastal flood",
     "country" : "Kiribati",
     "region" : ["Tarawa"],
     "city" : None,
     "locationAnnotation" : ["In Kiribati, rough seas, combined with tidal movements prompted by Tropical Cyclone Pam, resulted in widespread coastal flooding with extensive damage in the Kiribati capital, Tarawa."
                            ],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Storm",
     "hazardSubtypes" : "tropical storm",
     "country" : "Papua New Guinea",
     "region" : None,
     "city" : ["Port Moresby"],
     "locationAnnotation" : ["In Papua New Guinea, two regions experienced flooding and some landslides due to the overall weather system of Cyclone Pam as well as Tropical Cyclone Nathan.",
                             "The National Capital District, the location of Papua New Guinea’s capital Port Moresby, was also affected."
                            ],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Pam",
    },
    {"hazardType": "Storm",
     "hazardSubtypes" : "tropical storm",
     "country" : "Papua New Guinea",
     "region" : None,
     "city" : ["Port Moresby"],
     "locationAnnotation" : ["In Papua New Guinea, two regions experienced flooding and some landslides due to the overall weather system of Cyclone Pam as well as Tropical Cyclone Nathan.",
                             "The National Capital District, the location of Papua New Guinea’s capital Port Moresby, was also affected."
                            ],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Nathan",
    },
    {"hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Papua New Guinea",
     "region" : None,
     "city" : ["Port Moresby"],
     "locationAnnotation" : ["In Papua New Guinea, two regions experienced flooding and some landslides due to the overall weather system of Cyclone Pam as well as Tropical Cyclone Nathan.",
                             "The National Capital District, the location of Papua New Guinea’s capital Port Moresby, was also affected."
                            ],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Mass Movement",
     "hazardSubtypes" : "landslide",
     "country" : "Papua New Guinea",
     "region" : None,
     "city" : ["Port Moresby"],
     "locationAnnotation" : ["In Papua New Guinea, two regions experienced flooding and some landslides due to the overall weather system of Cyclone Pam as well as Tropical Cyclone Nathan.",
                             "The National Capital District, the location of Papua New Guinea’s capital Port Moresby, was also affected."
                            ],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Storm",
     "hazardSubtypes" : "tropical storm",
     "country" : "Solomon Islands",
     "region" : ["Temotu", "Malaita"],
     "city" : None,
     "locationAnnotation" : ["In Solomon Islands, Tropical Cyclone Pam created strong winds, heavy rainfall, and storm surges that impacted the provinces of Temotu and Malaita."
                            ],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Pam",
    },
    {"hazardType": "Storm",
     "hazardSubtypes" : "storm surge",
     "country" : "Solomon Islands",
     "region" : ["Temotu", "Malaita"],
     "city" : None,
     "locationAnnotation" : ["In Solomon Islands, Tropical Cyclone Pam created strong winds, heavy rainfall, and storm surges that impacted the provinces of Temotu and Malaita."
                            ],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Storm",
     "hazardSubtypes" : "tropical storm",
     "country" : "Tuvalu",
     "region" : None,
     "city" : None,
     "locationAnnotation" : ["In Tuvalu, the effects of Tropical Cyclone Pam have caused sea swells, storm surges and saltwater intrusion on eight islands."
                            ],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Pam",
    },
    {"hazardType": "Storm",
     "hazardSubtypes" : "storm surge",
     "country" : "Tuvalu",
     "region" : ["Funafuti", "Nanumea", "Nanumaga", "Niutao", "Nui", "Vaitupu", "Nukufetau", "Nukulaelae atolls"],
     "city" : None,
     "locationAnnotation" : ["Prolonged sea swells and storm surges impacted Funafuti, Nanumea, Nanumaga, Niutao, Nui, Vaitupu, Nukufetau and Nukulaelae atolls."
                            ],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Pam",
    }
]

df = pd.DataFrame(dict_)
df['appealCode'] = "MDR55001"
df['reportDate'] = "15/05/2015"
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

df_combined_labelled = add_new_labelled_report(df_combined_labelled, df)

In [ ]:
i = 1
appealCode_i = appealCode_missing[i]
found = False
for report in filtered_reports :
    if report['appealCode']==appealCode_i :
        print(report['date'])
        if not found :
            for sent in report['nathaz_text'] :
                print(sent)
            found = True
        # for sent in report['nathaz_text'] :
        #     print(sent)
if appealCode_i in reports_labelled.appealCode.unique() :
    print(reports_labelled.loc[reports_labelled.appealCode==appealCode_i])

In [16]:
dict_=[
    {"hazardType": "Earthquake",
     "hazardSubtypes" : "ground movement",
     "country" : "Indonesia",
     "region" : ["Lombok", "West Nusa Tenggara"],
     "city" : None,
     "locationAnnotation" : ["SITUATION ANALYSIS Appeal History 29 July 2018: A 6.4 magnitude earthquake strikes off Lombok, province of West Nusa Tenggara 31 July: IFRC allocates CHF 211,569 from the Disaster Relief Emergency Fund (DREF) to enable PMI to meet the humanitarian needs of 1,000 households (4,000 people)"
                            ],
     "startYear" : 2018,
     "startMonth" : 7,
     "startDay" : 29,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Earthquake",
     "hazardSubtypes" : "ground movement",
     "country" : "Indonesia",
     "region" : ["Lombok"],
     "city" : None,
     "locationAnnotation" : ["5 August: A second and stronger earthquake, of 7.0 magnitude and depth of 15km hits Lombok 7 August: An Emergency Appeal seeking CHF 8.9 million is launched to support PMI in providing assistance to 20,000 households for 18 months."
                            ],
     "startYear" : 2018,
     "startMonth" : 8,
     "startDay" : 5,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Earthquake",
     "hazardSubtypes" : "ground movement",
     "country" : "Indonesia",
     "region" : ["Lombok"],
     "city" : None,
     "locationAnnotation" : ["9 and 18 August: New 5.9 and 6.4 magnitude earthquakes strike Lombok."
                            ],
     "startYear" : 2018,
     "startMonth" : 8,
     "startDay" : 9,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Earthquake",
     "hazardSubtypes" : "ground movement",
     "country" : "Indonesia",
     "region" : ["Lombok"],
     "city" : None,
     "locationAnnotation" : ["9 and 18 August: New 5.9 and 6.4 magnitude earthquakes strike Lombok."
                            ],
     "startYear" : 2018,
     "startMonth" : 8,
     "startDay" : 18,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Earthquake",
     "hazardSubtypes" : "ground movement",
     "country" : "Indonesia",
     "region" : ["Central Sulawesi"],
     "city" : None,
     "locationAnnotation" : ["28 September: A 7.4 magnitude earthquake at a depth of 10km strikes Central Sulawesi, followed by a tsunami which hit coastal areas of Donggala and Palu regencies."
                            ],
     "startYear" : 2018,
     "startMonth" : 9,
     "startDay" : 28,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Earthquake",
     "hazardSubtypes" : "tsunami",
     "country" : "Indonesia",
     "region" : ["Donggala"],
     "city" : ["Palu"],
     "locationAnnotation" : ["28 September: A 7.4 magnitude earthquake at a depth of 10km strikes Central Sulawesi, followed by a tsunami which hit coastal areas of Donggala and Palu regencies.",
                             "The strongest of which measured at 7.4 magnitude and 10km deep with the epicentre in Donggala Regency, close to the provincial capital Palu."
                            ],
     "startYear" : 2018,
     "startMonth" : 9,
     "startDay" : 28,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Earthquake",
     "hazardSubtypes" : "tsunami",
     "country" : "Indonesia",
     "region" : ["Sunda Strait", "South Lampung"],
     "city" : ["Pandeglang", "Serang"],
     "locationAnnotation" : ["22 December: Coastal areas around the Sunda Strait, specifically in Pandeglang, South Lampung and Serang districts are hit by waves caused by a massive landslide on Mount Kakatoa, an active volcano in the center of the strait."
                            ],
     "startYear" : 2018,
     "startMonth" : 12,
     "startDay" : 22,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Mass Movement",
     "hazardSubtypes" : "landslide",
     "country" : "Indonesia",
     "region" : ["Mount Kakatoa"],
     "city" : None,
     "locationAnnotation" : ["22 December: Coastal areas around the Sunda Strait, specifically in Pandeglang, South Lampung and Serang districts are hit by waves caused by a massive landslide on Mount Kakatoa, an active volcano in the center of the strait."
                            ],
     "startYear" : 2018,
     "startMonth" : 12,
     "startDay" : 22,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
]


df = pd.DataFrame(dict_)
df['appealCode'] = "MDRID013"
df['reportDate'] = "20/09/2021"
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

df_combined_labelled = add_new_labelled_report(df_combined_labelled, df)

In [ ]:
i = 2
appealCode_i = appealCode_missing[i]
found = False
for report in filtered_reports :
    if report['appealCode']==appealCode_i :
        print(report['date'])
        if not found :
            for sent in report['nathaz_text'] :
                print(sent)
            found = True
        # for sent in report['nathaz_text'] :
        #     print(sent)
if appealCode_i in reports_labelled.appealCode.unique() :
    print(reports_labelled.loc[reports_labelled.appealCode==appealCode_i])

In [17]:
dict_=[
    {"hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Brazil",
     "region" : ["Bahía", "Minas Gerais", "south-eastern"],
     "city" : ["Jucuruçu", "Medeiros Neto", "Petrópolis", "Angra dos Reis", "Paraty", "Itamaraju"],
     "locationAnnotation" : ["1 DREF Operation Nº MDRBR010 Glide N° FL-2021-000204-BRA Operation start date: 16 December 2021 Timeframe: 5 months (2-month extension) Operation end date: 31 May 2022 Final Report publication: 24 May 2024 Reporting period covered by this update: 16 December 2021 – 31 May 2022 IFRC Category allocated to the of the disaster or crisis: Yellow DREF allocated: 342,866 Swiss francs (an increase from the original CHF 261,223) Total number of people affected: Over 1 million in both states Number of people to be assisted: 4,000 (800 families) States affected: Bahia, Minas Gerais Areas targeted: Bahía (Jucuruçu and Medeiros Neto) Minas Gerais Host National Society presence: The Brazilian Red Cross (BRC) has its national headquarters in Rio de Janeiro and 21 branches with 6,000 volunteers and 300 staff members.",
                             "In mid-February and early March, further rains affected other areas of the country (Petrópolis, Angra dos Reis, Paraty)",
                             "Heavy rainfall, including the passage of a subtropical cyclone over Bahia on 7 December 2021, caused flooding and landslides in south- eastern Brazil.",
                             "The main affected areas in the far south are in the municipalities of Medeiros Neto, Jucuruçu and Itamaraju, where the Brazilian Red Cross (CRB) carried out its DREF Final Report Brazil: Floods 2 response."
                            ],
     "startYear" : 2021,
     "startMonth" : 11,
     "startDay" : None,
     "endYear" : 2022,
     "endMonth" : 5,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Storm",
     "hazardSubtypes" : "tropical storm",
     "country" : "Brazil",
     "region" : ["Bahía"],
     "city" : None,
     "locationAnnotation" : ["Heavy rainfall, including the passage of a subtropical cyclone over Bahia on 7 December 2021, caused flooding and landslides in south- eastern Brazil."
                            ],
     "startYear" : 2021,
     "startMonth" : 12,
     "startDay" : 7,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Mass Movement",
     "hazardSubtypes" : "landslide",
     "country" : "Brazil",
     "region" : ["south-eastern"],
     "city" : None,
     "locationAnnotation" : ["Heavy rainfall, including the passage of a subtropical cyclone over Bahia on 7 December 2021, caused flooding and landslides in south- eastern Brazil."
                            ],
     "startYear" : 2021,
     "startMonth" : 12,
     "startDay" : 7,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Mass Movement",
     "hazardSubtypes" : "rockfall",
     "country" : "Brazil",
     "region" : None,
     "city" : None,
     "locationAnnotation" : None,
     "startYear" : 2021,
     "startMonth" : 11,
     "startDay" : None,
     "endYear" : 2022,
     "endMonth" : 5,
     "endDay" : None,
     "hazardName" : None,
    },
]


df = pd.DataFrame(dict_)
df['appealCode'] = "MDRBR010"
df['reportDate'] = "24/05/2024"
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

df_combined_labelled = add_new_labelled_report(df_combined_labelled, df)

In [ ]:
i = 3
appealCode_i = appealCode_missing[i]
found = False
for report in filtered_reports :
    if report['appealCode']==appealCode_i :
        print(report['date'])
        if not found :
            for sent in report['nathaz_text'] :
                print(sent)
            found = True
        # for sent in report['nathaz_text'] :
        #     print(sent)
if appealCode_i in reports_labelled.appealCode.unique() :
    print(reports_labelled.loc[reports_labelled.appealCode==appealCode_i])

In [18]:
dict_=[
    {"hazardType": "Drought",
     "hazardSubtypes" : "drought",
     "country" : "Syria",
     "region" : ["Northern", "North-East"],
     "city" : ["Ar-Raqqa", "Deir-ez-Zor"],
     "locationAnnotation" : ["Low and erratic rainfall during the 2020/2021 winter season, accompanied by higher-than-average temperatures, led to drought conditions in Northern and North-East Syria, as well as in other parts of Syria, with significant crop and livestock production losses.",
                             "The combined effect of reduced water levels in the Euphrates river and drought conditions impacted the food and nutrition security of households dependent on agriculture in Ar-Raqqa and Deir-ez-Zor."
                            ],
     "startYear" : 2020,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : 2021,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    }
]
df = pd.DataFrame(dict_)
df['appealCode'] = "MDRSY006"
df['reportDate'] = "06/10/2022"
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

df_combined_labelled = add_new_labelled_report(df_combined_labelled, df)

In [ ]:
i = 4
appealCode_i = appealCode_missing[i]
found = False
for report in filtered_reports :
    if report['appealCode']==appealCode_i :
        print(report['date'])
        if not found :
            for sent in report['nathaz_text'] :
                print(sent)
            found = True
        # for sent in report['nathaz_text'] :
        #     print(sent)
if appealCode_i in reports_labelled.appealCode.unique() :
    print(reports_labelled.loc[reports_labelled.appealCode==appealCode_i])

In [19]:
dict_=[
    {"hazardType": "Wildfire",
     "hazardSubtypes" : "forest fire",
     "country" : "Algeria",
     "region" : None,
     "city" : ["Bejaia", "Jijel", "Souk Ahras", "El Taref", "Setif", "Skikda", "tipaza", "tizi-ouzou", "Guelma", "Batna", "Mila", "Annaba", "Constantine", "Bordj Bou Arrerid"],
     "locationAnnotation" : ["SITUATION ANALYSIS Description of the disaster More than 100 fires raged in north-eastern Algeria on the night of Sunday, August 14, 2022, affecting 14 governorates (Wilayas): Bejaia, Jijel, Souk Ahras, El Taref, Setif, Skikda, tipaza, tizi-ouzou, Guelma, Batna, Mila, Annaba, Constantine and Bordj Bou Arrerid.",
                             "This year's fires have resumed in some Wilayas mentioned above such as Bejaia, Jijel, Setif, Skikda, Tizi-Ouzou, Guelma and Bordj Bou Arréridj"
                            ],
     "startYear" : 2022,
     "startMonth" : 8,
     "startDay" : 14,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Wildfire",
     "hazardSubtypes" : "forest fire",
     "country" : "Algeria",
     "region" : None,
     "city" : ["Tizi Ouzou", "Béjaïa", "Bouira", "Sétif", "Jijel", "Boumerdès", "Bordj Bou Arréridj", "Blida", "Médéa", "Khenchela", "Guelma", "Tébessa", "Tiaret", "Skikda"],
     "locationAnnotation" : ["The 2021 fires broke out in the Wilayas of Tizi Ouzou, Béjaïa, Bouira, Sétif, Jijel, Boumerdès, Bordj Bou Arréridj, Blida, Médéa, Khenchela, Guelma, Tébessa, Tiaret and Skikda."
                            ],
     "startYear" : 2021,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
]
df = pd.DataFrame(dict_)
df['appealCode'] = "MDRDZ008"
df['reportDate'] = "24/11/2023"
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

df_combined_labelled = add_new_labelled_report(df_combined_labelled, df)

In [ ]:
i = 5
appealCode_i = appealCode_missing[i]
found = False
for report in filtered_reports :
    if report['appealCode']==appealCode_i :
        print(report['date'])
        if not found :
            for sent in report['nathaz_text'] :
                print(sent)
            found = True
        # for sent in report['nathaz_text'] :
        #     print(sent)
if appealCode_i in reports_labelled.appealCode.unique() :
    print(reports_labelled.loc[reports_labelled.appealCode==appealCode_i])

In [20]:
dict_=[
    {"hazardType": "Mass Movement",
     "hazardSubtypes" : "landslide",
     "country" : "Cameroon",
     "region" : None,
     "city" : ["Yaoundé"],
     "locationAnnotation" : ["On the night of Sunday 08 October 2023, torrential rain caused a landslide in the Mbankolo neighborhood in the Yaoundé II district council following the collapse of the embankment of an artiﬁcial lake uphill."
                            ],
     "startYear" : 2023,
     "startMonth" : 10,
     "startDay" : 8,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    }
]
df = pd.DataFrame(dict_)
df['appealCode'] = "MDRCM036"
df['reportDate'] = "23/10/2023"
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

df_combined_labelled = add_new_labelled_report(df_combined_labelled, df)

In [ ]:
i = 6
appealCode_i = appealCode_missing[i]
found = False
for report in filtered_reports :
    if report['appealCode']==appealCode_i :
        print(report['date'])
        if not found :
            for sent in report['nathaz_text'] :
                print(sent)
            found = True
        # for sent in report['nathaz_text'] :
        #     print(sent)
if appealCode_i in reports_labelled.appealCode.unique() :
    print(reports_labelled.loc[reports_labelled.appealCode==appealCode_i])

In [21]:
dict_=[
    {"hazardType": "Extreme temperature",
     "hazardSubtypes" : "coldwave",
     "country" : "Afghanistan",
     "region" : ["Badakhshan", "Badghis", "Faryab", "Ghor", "Helmand", "Jawzjan", "Nuristan", "Nangarhar", "Uruzgan", "Zabul"],
     "city" : ["Balkh", "Bamyan", "Farah", "Herat", "Kandahar", "Kunduz", "Sar-e-Pul"],
     "locationAnnotation" : ["Page 1 / 18 DREF Operational Update Afghanistan Cold Wave 2024 ARCS conducting beneficiary registration in Nangarhar Province (Photo: ARCS) Appeal: MDRAF014 Total DREF Allocation: - Crisis Category: Yellow Hazard: Cold Wave Glide Number: CW-2024-000025-AFG People Affected: 325,205 people People Targeted: 22,400 people Event Onset: Sudden Operation Start Date: 16-03-2024 New Operational End Date: 31-07-2024 Total Operating Timeframe: 4 months Reporting Timeframe Start Date: 16-03-2024 Reporting Timeframe End Date: 15-04-2024 Additional Allocation Requested: - Targeted Areas: Badakhshan, Badghis, Balkh, Bamyan, Farah, Faryab, Ghor, Helmand, Herat, Jawzjan, Kandahar, Kunduz Page 2 / 18 Description of the Event Afghanistan map that highlights 11 affected provinces Date of event 2024-03-03 What happened, where and when?",
                             "The provinces most affected include Badakhshan, Badghis, Balkh, Farah, Faryab, Ghor, Herat, Jawzjan, Kunduz, Nuristan, Nangarhar, Sar-e-Pul, Uruzgan, and Zabul."
                            ],
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    }
]
df = pd.DataFrame(dict_)
df['appealCode'] = "MDRAF014"
df['reportDate'] = "25/04/2024"
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

df_combined_labelled = add_new_labelled_report(df_combined_labelled, df)

In [ ]:
i = 7
appealCode_i = appealCode_missing[i]
found = False
for report in filtered_reports :
    if report['appealCode']==appealCode_i :
        print(report['date'])
        if not found :
            for sent in report['nathaz_text'] :
                print(sent)
            found = True
        # for sent in report['nathaz_text'] :
        #     print(sent)
if appealCode_i in reports_labelled.appealCode.unique() :
    print(reports_labelled.loc[reports_labelled.appealCode==appealCode_i])

In [22]:
dict_=[
    {"hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Uganda",
     "region" : ["eastern"],
     "city" : ["Mbale", "Kapchorwa", "Bulambuli", "Bukedea", "Butaleja", "Sironko", "Bududa", "Namisindwa", "Ntoroko"],
     "locationAnnotation" : ["In April 2024, the Eastern Uganda-Elgon region experienced heavy rainfall, as forecasted by the Uganda National Meteorological Authority (UNMA)",
                             "This resulted in significant impacts from episodic floods, hailstorms, and landslides in various areas, including Mbale, Kapchorwa, Bulambuli, Bukedea, Butaleja, Sironko, Bududa, and Namisindwa.",
                             "Further, Ntoroko district witnessed its ever-recorded highest levels of flooding in August whose assessment reveals that 2,355 households comprising of 11,775 people have been greatly impacted."
                            ],
     "startYear" : 2024,
     "startMonth" : 4,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Storm",
     "hazardSubtypes" : "hail",
     "country" : "Uganda",
     "region" : ["eastern"],
     "city" : ["Mbale", "Kapchorwa", "Bulambuli", "Bukedea", "Butaleja", "Sironko", "Bududa", "Namisindwa"],
     "locationAnnotation" : ["This resulted in significant impacts from episodic floods, hailstorms, and landslides in various areas, including Mbale, Kapchorwa, Bulambuli, Bukedea, Butaleja, Sironko, Bududa, and Namisindwa."
                            ],
     "startYear" : 2024,
     "startMonth" : 4,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Mass Movement",
     "hazardSubtypes" : "landslide",
     "country" : "Uganda",
     "region" : ["eastern"],
     "city" : ["Mbale", "Kapchorwa", "Bulambuli", "Bukedea", "Butaleja", "Sironko", "Bududa", "Namisindwa"],
     "locationAnnotation" : ["In April 2024, the Eastern Uganda-Elgon region experienced heavy rainfall, as forecasted by the Uganda National Meteorological Authority (UNMA)",
                             "This resulted in significant impacts from episodic floods, hailstorms, and landslides in various areas, including Mbale, Kapchorwa, Bulambuli, Bukedea, Butaleja, Sironko, Bududa, and Namisindwa."
                            ],
     "startYear" : 2024,
     "startMonth" : 4,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
]
df = pd.DataFrame(dict_)
df['appealCode'] = "MDRUG050"
df['reportDate'] = "30/08/2024"
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

df_combined_labelled = add_new_labelled_report(df_combined_labelled, df)

In [ ]:
i = 8
appealCode_i = appealCode_missing[i]
found = False
for report in filtered_reports :
    if report['appealCode']==appealCode_i :
        print(report['date'])
        if not found :
            for sent in report['nathaz_text'] :
                print(sent)
            found = True
        # for sent in report['nathaz_text'] :
        #     print(sent)
if appealCode_i in reports_labelled.appealCode.unique() :
    print(reports_labelled.loc[reports_labelled.appealCode==appealCode_i])

In [23]:
dict_=[
    {"hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Rwanda",
     "region" : ["western", "northern", "southern"],
     "city" : None,
     "locationAnnotation" : ["According to assessments carried out by the Rwanda Red Cross and other stake holders and MINEMA led, the western, northern and southern provinces of Rwanda were the areas hardest hit by the flooding."
                            ],
     "startYear" : 2023,
     "startMonth" : 5,
     "startDay" : 1,
     "endYear" : 2023,
     "endMonth" : 6,
     "endDay" : None,
     "hazardName" : None,
    }
]
df = pd.DataFrame(dict_)
df['appealCode'] = "MDRRW022"
df['reportDate'] = "05/09/2024"
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

df_combined_labelled = add_new_labelled_report(df_combined_labelled, df)

In [ ]:
i = 9
appealCode_i = appealCode_missing[i]
found = False
for report in filtered_reports :
    if report['appealCode']==appealCode_i :
        print(report['date'])
        if not found :
            for sent in report['nathaz_text'] :
                print(sent)
            found = True
        # for sent in report['nathaz_text'] :
        #     print(sent)
if appealCode_i in reports_labelled.appealCode.unique() :
    print(reports_labelled.loc[reports_labelled.appealCode==appealCode_i])

In [26]:
dict_=[
    {"hazardType": "Storm",
     "hazardSubtypes" : "tropical storm",
     "country" : "Mozambique",
     "region" : None,
     "city" : None,
     "locationAnnotation" : None,
     "startYear" : 2024,
     "startMonth" : 3,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Filippo",
    },
    {"hazardType": "Storm",
     "hazardSubtypes" : "tropical storm",
     "country" : "Mozambique",
     "region" : None,
     "city" : None,
     "locationAnnotation" : None,
     "startYear" : 2023,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : "Freddy",
    },
    {"hazardType": "Drought",
     "hazardSubtypes" : "drought",
     "country" : "Mozambique",
     "region" : ["Tete", "Gaza", "Manica", "Inhambane", "southern", "central"],
     "city" : None,
     "locationAnnotation" : ["Provinces such as Tete, Gaza, Manica, and Inhambane, known for high production and pastoral activities, have seen significant reductions in agricultural output with well below average harvests compared to last year and the fiveyear average.",
                             "SITUATION ANALYSIS Description of the crisis Mozambique is currently experiencing severe effects from the strong 20232024 El Nio season which brought below average rainfall to southern and central Mozambique and above average rainfall to the northern regions, severely impacting agriculture and rural livelihoods."],
     "startYear" : 2024,
     "startMonth" : 4,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    }
]
df = pd.DataFrame(dict_)
df['appealCode'] = "MDRMZ024"
df['reportDate'] = "30/08/2024"
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

df_combined_labelled = add_new_labelled_report(df_combined_labelled, df)

In [ ]:
i = 10
appealCode_i = appealCode_missing[i]
found = False
for report in filtered_reports :
    if report['appealCode']==appealCode_i :
        print(report['date'])
        if not found :
            for sent in report['nathaz_text'] :
                print(sent)
            found = True
        # for sent in report['nathaz_text'] :
        #     print(sent)
if appealCode_i in reports_labelled.appealCode.unique() :
    print(reports_labelled.loc[reports_labelled.appealCode==appealCode_i])

In [28]:
dict_=[
    {"hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Zambia",
     "region" : ["Lusaka", "Luapula", "Western", "Southern", "Central", "Northwestern"],
     "city" : ["Mazabuka"],
     "locationAnnotation" : ["This includes more than three million children under 18 years of age, mostly based in the provinces of Lusaka, Luapula, and the Western, Southern, Central, and Northwestern Provinces.",
                             "Page 1 21 DREF Operation Zambia Drought 2024 Staff checking on a maize field affected by drought in Mazabuka District, Southern Province Appeal MDRZM022 Country Zambia Hazard Drought Type of DREF Response Crisis Category Orange Event Onset Slow DREF Allocation CHF 750,459 Glide Number People Affected 5,000,000 people People Targeted 160,000 people Operation Start Date 20240322 Operation Timeframe 6 months Operation End Date 30092024 DREF Published 28032024 Targeted Areas Southern Page 2 21 Description of the Event Date when the trigger was met 20240229 Districts affeccted by Drought What happened, where and when?",
                             "The provinces affected include NorthWestern, Southern, Western, Central and Eastern."
                            ],
     "startYear" : 2024,
     "startMonth" : 2,
     "startDay" : 29,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    }
]
df = pd.DataFrame(dict_)
df['appealCode'] = "MDRZM022"
df['reportDate'] = "30/08/2024"
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

df_combined_labelled = add_new_labelled_report(df_combined_labelled, df)

In [ ]:
i = 11
appealCode_i = appealCode_missing[i]
found = False
for report in filtered_reports :
    if report['appealCode']==appealCode_i :
        print(report['date'])
        if not found :
            for sent in report['nathaz_text'] :
                print(sent)
            found = True
        # for sent in report['nathaz_text'] :
        #     print(sent)
if appealCode_i in reports_labelled.appealCode.unique() :
    print(reports_labelled.loc[reports_labelled.appealCode==appealCode_i])

In [30]:
dict_=[
    {"hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Nigeria",
     "region" : ["Bauchi", "Kebbi", "Sokoto", "Zamfara"],
     "city" : ["Kano", "Maiduguri", "Giade", "Shira", "Katagum"],
     "locationAnnotation" : ["DREF Operation Nigeria Floods DREF 2024 Flood cuts off major access road linking Kano to Maiduguri in Katagum community, Bauchi state Appeal MDRNG041 Country Nigeria Hazard Flood Type of DREF Response Crisis Category Yellow Event Onset Sudden DREF Allocation CHF 231,293 Glide Number People Affected 50,000 people People Targeted 9,000 people Operation Start Date 03092024 Operation Timeframe 4 months Operation End Date 31012025 DREF Published 06092024 Targeted Areas Bauchi, Kebbi, Sokoto, Zamfara Page 1 17 Description of the Event Date of event 13082024 Nigeria Flood Forecast 2024 What happened, where and when?",
                             "From August 8 to August 13, 2024, continuous heavy rainfall triggered severe flooding across Nigeria, leading to widespread devastation and displacement in states such as Bauchi, Sokoto, and Zamfara.",
                             "In Bauchi State, over 1,000 homes were destroyed, particularly impacting the Giade, Shira, and Katagum local government areas."
                            ],
     "startYear" : 2024,
     "startMonth" : 8,
     "startDay" : 8,
     "endYear" : 2024,
     "endMonth" : 8,
     "endDay" : 13,
     "hazardName" : None,
    },
    {"hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Nigeria",
     "region" : ["Sokoto"],
     "city" : ["Gada", "Dantudu", "Balakozo", "Gidan Tudu", "Tsitse"],
     "locationAnnotation" : ["Earlier, on July 17, 2024, flooding in Sokoto State displaced 1,664 people and caused extensive damage to farmlands and livestock across four communities in Gada Local Government Area, including Dantudu, Balakozo, Gidan Tudu, and Tsitse."
                            ],
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : 17,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    }
]
df = pd.DataFrame(dict_)
df['appealCode'] = "MDRNG041"
df['reportDate'] = "30/08/2024"
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

df_combined_labelled = add_new_labelled_report(df_combined_labelled, df)

In [ ]:
i = 12
appealCode_i = appealCode_missing[i]
found = False
for report in filtered_reports :
    if report['appealCode']==appealCode_i :
        print(report['date'])
        if not found :
            for sent in report['nathaz_text'] :
                print(sent)
            found = True
        # for sent in report['nathaz_text'] :
        #     print(sent)
if appealCode_i in reports_labelled.appealCode.unique() :
    print(reports_labelled.loc[reports_labelled.appealCode==appealCode_i])

In [32]:
dict_=[
    {"hazardType": "Flood",
     "hazardSubtypes" : "flash flood",
     "country" : "Sudan",
     "region" : ["Red Sea", "River Nile", "Northern State"],
     "city" : None,
     "locationAnnotation" : ["So far, Red Sea, River Nile, and Northern State have been the most severely affected."
                            ],
     "startYear" : 2024,
     "startMonth" : 6,
     "startDay" : 1,
     "endYear" : 2024,
     "endMonth" : 8,
     "endDay" : 12,
     "hazardName" : None,
    },
    {"hazardType": "Flood",
     "hazardSubtypes" : "riverine flood",
     "country" : "Sudan",
     "region" : ["Red Sea", "River Nile", "Northern State"],
     "city" : None,
     "locationAnnotation" : ["So far, Red Sea, River Nile, and Northern State have been the most severely affected."
                            ],
     "startYear" : 2024,
     "startMonth" : 6,
     "startDay" : 1,
     "endYear" : 2024,
     "endMonth" : 8,
     "endDay" : 12,
     "hazardName" : None,
    },
]
df = pd.DataFrame(dict_)
df['appealCode'] = "MDRSD034"
df['reportDate'] = "06/09/2024"
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

df_combined_labelled = add_new_labelled_report(df_combined_labelled, df)

In [ ]:
i = 13
appealCode_i = appealCode_missing[i]
found = False
for report in filtered_reports :
    if report['appealCode']==appealCode_i :
        print(report['date'])
        if not found :
            for sent in report['nathaz_text'] :
                print(sent)
            found = True
        # for sent in report['nathaz_text'] :
        #     print(sent)
if appealCode_i in reports_labelled.appealCode.unique() :
    print(reports_labelled.loc[reports_labelled.appealCode==appealCode_i])

In [33]:
dict_=[
    {"hazardType": "Flood",
     "hazardSubtypes" : "riverine flood",
     "country" : "Benin",
     "region" : ["Couffo", "Adoukandji", "Ahomadegbe", "Gnizounme", "Tchito", "Tohou", "Zalli"],
     "city" : ["Mono", "Couffo", "Zou", "Oum", "Ahouada", "Hazin", "Yamontou", "Ahomadegbe", "Gnizounme", "Hangbannou", "Tandji", "Aboti", "Zounhome", "Hehokpa", "Sawanou", "Tohou Centre", "Adjassagon"],
     "locationAnnotation" : ["Intense rainfall observed in the departments of Mono, Couffo, Zou and Oum in the South of Benin caused the overflow of the river Couffo on 26 June 2024 in 6 of the 11 districts of the commune it crosses in Couffo department, Adoukandji, Ahomadegbe, Gnizounme, Tchito, Tohou and Zalli.",
                             "A rapid assessment conducted during the following days by Benin Red Cross and the Lalo council on July 1, 2024 indicates that about 13 villages Ahouada, Hazin, Yamontou, Ahomadegbe, Gnizounme, Hangbannou, Tandji, Aboti, Zounhome, Hehokpa, Sawanou, Tohou Centre and Adjassagon were flooded with several houses destroyed and damaged.",
                            ],
     "startYear" : 2024,
     "startMonth" : 6,
     "startDay" : 26,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
]
df = pd.DataFrame(dict_)
df['appealCode'] = "MDRBJ019"
df['reportDate'] = "07/09/2024"
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

df_combined_labelled = add_new_labelled_report(df_combined_labelled, df)

In [ ]:
i = 14
appealCode_i = appealCode_missing[i]
found = False
for report in filtered_reports :
    if report['appealCode']==appealCode_i :
        print(report['date'])
        if not found :
            for sent in report['nathaz_text'] :
                print(sent)
            found = True
        # for sent in report['nathaz_text'] :
        #     print(sent)
if appealCode_i in reports_labelled.appealCode.unique() :
    print(reports_labelled.loc[reports_labelled.appealCode==appealCode_i])

In [34]:
dict_=[
    {"hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Cameroon",
     "region" : ["Cameroons Far North region", "Logone", "Chari", "Mayo Danay", "Diamar"],
     "city" : ["Blangoua", "Mackary", "Zina", "Maga", "Yagoua", "Ndoukoula", "Mokolo"],
     "locationAnnotation" : ["Series of floods have been recorded since August 19, reaching critical levels in the Logone et Chari and Mayo Danay divisions between August 11 and 21, 2024.",
                             "The most affected districts are Blangoua, Mackary, and Zina in the Logone and Chari department, and Maga, Yagoua in the Mayo Danay division.",
                             "In the Logone et Chari division, the affected districts are Blangoua with nearly 75,000 people affected Makary with 43,000 Zina with 9,000 people affected In the Mayo Danay division Maga with 18,000 people affected Yagoua with nearly 13,000 people The rains continue with weather forecasts predicting more significant impacts in the divisions already mentioned Page 2 19 above, as well as in others that have also been experiencing heavy rainfall for several days.",
                             "Notably, in the Diamar division, where Ndoukoula district has reported over 400 people affected to date, while in Mayo Tsanaga, Mokolo district, has recorded nearly 200 affected people."
                            ],
     "startYear" : 2024,
     "startMonth" : 8,
     "startDay" : 10,
     "endYear" : 2024,
     "endMonth" : 8,
     "endDay" : 28,
     "hazardName" : None,
    },
]
df = pd.DataFrame(dict_)
df['appealCode'] = "MDRCM039"
df['reportDate'] = "13/09/2024"
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

df_combined_labelled = add_new_labelled_report(df_combined_labelled, df)

In [ ]:
i = 15
appealCode_i = appealCode_missing[i]
found = False
for report in filtered_reports :
    if report['appealCode']==appealCode_i :
        print(report['date'])
        if not found :
            for sent in report['nathaz_text'] :
                print(sent)
            found = True
        # for sent in report['nathaz_text'] :
        #     print(sent)
if appealCode_i in reports_labelled.appealCode.unique() :
    print(reports_labelled.loc[reports_labelled.appealCode==appealCode_i])

In [35]:
dict_=[
    {"hazardType": "Flood",
     "hazardSubtypes" : "flash flood",
     "country" : "Pakistan",
     "region" : ["Balochistan ", "Sindh ", "Punjab", "Khyber Pakhtunkhwa", "Azad Jammu"],
     "city" : ["Jacobabad", "Naushahro Feroz", "Ghotki", "Sukkur", "Sanghar", "Dadu", "Shaheed Benazirabad", "Kashmor", "Taluka Tando Adam"],
     "locationAnnotation" : ["Regionally, Balochistan received 239 per cent more rainfall than usual, Sindh 318 per cent, Punjab 111 per cent, and Khyber Pakhtunkhwa KP 25 per cent.",
                             "This unprecedented volume of rainfall, coupled with unusually high temperatures, accelerated snowmelt in KP, Azad Jammu and Kashmir AJK, and Gilgit Baltistan GB, leading to catastrophic flash floods and landslides.",
                             "Areas such as Jacobabad, Naushahro Feroz, Ghotki, Sukkur, Sanghar, Dadu, Shaheed Benazirabad, and Kashmor have faced heavy rains, resulting in substantial damage to homes and infrastructure.",
                             "In district Sanghar, the Deputy Commissioner reported a massive breach in the Rohri Canal that created numerous water bodies in Taluka Tando Adam, inundating over 35 villages and displacing 9,500 people who are now residing in relief camps."
                            ],
     "startYear" : 2024,
     "startMonth" : 7,
     "startDay" : None,
     "endYear" : 2024,
     "endMonth" : 9,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Mass movement",
     "hazardSubtypes" : "landslide",
     "country" : "Pakistan",
     "region" : ["Khyber Pakhtunkhwa", "Azad Jammu and Kashmir", "Gilgit Baltistan"],
     "city" : None,
     "locationAnnotation" : ["This unprecedented volume of rainfall, coupled with unusually high temperatures, accelerated snowmelt in KP, Azad Jammu and Kashmir AJK, and Gilgit Baltistan GB, leading to catastrophic flash floods and landslides"],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    },
    {"hazardType": "Extreme temperature",
     "hazardSubtypes" : "heatwave",
     "country" : "Pakistan",
     "region" : ["Khyber Pakhtunkhwa", "Azad Jammu and Kashmir", "Gilgit Baltistan"],
     "city" : None,
     "locationAnnotation" : ["This unprecedented volume of rainfall, coupled with unusually high temperatures, accelerated snowmelt in KP, Azad Jammu and Kashmir AJK, and Gilgit Baltistan GB, leading to catastrophic flash floods and landslides."],
     "startYear" : None,
     "startMonth" : None,
     "startDay" : None,
     "endYear" : None,
     "endMonth" : None,
     "endDay" : None,
     "hazardName" : None,
    }
]
df = pd.DataFrame(dict_)
df['appealCode'] = "MDRPK026"
df['reportDate'] = "17/09/2024"
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

df_combined_labelled = add_new_labelled_report(df_combined_labelled, df)

In [ ]:
i = 16
appealCode_i = appealCode_missing[i]
found = False
for report in filtered_reports :
    if report['appealCode']==appealCode_i :
        print(report['date'])
        if not found :
            for sent in report['nathaz_text'] :
                print(sent)
            found = True
        # for sent in report['nathaz_text'] :
        #     print(sent)
if appealCode_i in reports_labelled.appealCode.unique() :
    print(reports_labelled.loc[reports_labelled.appealCode==appealCode_i])

In [44]:
dict_=[
    {"hazardType": "Flood",
     "hazardSubtypes" : None,
     "country" : "Algeria",
     "region" : ["southern and western Algeria"],
     "city" : ["Bchar", "Elbayadh", "Beni Abbes", "Tamanrasset", "Tiaret", "Tindouf", "Naama"],
     "locationAnnotation" : ["The most affected areas include Bchar, Elbayadh, Beni Abbes, Tamanrasset, Tiaret, Tindouf, and Naama."],
     "startYear" : 2024,
     "startMonth" : 9,
     "startDay" : 5,
     "endYear" : 2024,
     "endMonth" : 9,
     "endDay" : 8,
     "hazardName" : None,
    }
]
df = pd.DataFrame(dict_)
df['appealCode'] = "MDRDZ011"
df['reportDate'] = "22/09/2024"
df['Country'] = [country_name_to_iso3(cntr) for cntr in df['country']]

df_combined_labelled = add_new_labelled_report(df_combined_labelled, df)

In [41]:
df_combined_labelled.to_csv(DATA_LABELLED+save_name, index=False)

,hazardType,hazardSubtypes,country,region,city,locationAnnotation,startYear,startMonth,startDay,endYear,endMonth,endDay,hazardName,appealCode,Country,reportDate
0,Storm,tropical storm,Vanuatu,None,None,[The Vanuatu Red Cross Society (VRCS) has mobi...,NaN,NaN,NaN,NaN,NaN,NaN,Pam,MDR55001,VUT,15/05/2015
1,Storm,tropical storm,Kiribati,"[Tamana, Arorae, Onotoa, Nonouti]",None,[The Vanuatu Red Cross Society (VRCS) has mobi...,NaN,NaN,NaN,NaN,NaN,NaN,Pam,MDR55001,KIR,15/05/2015
2,Storm,storm surge,Kiribati,"[Tamana, Arorae, Onotoa, Nonouti]",None,"[The four outer atolls of Tamana, Arorae, Onot...",NaN,NaN,NaN,NaN,NaN,NaN,None,MDR55001,KIR,15/05/2015
3,Flood,coastal flood,Kiribati,[Tarawa],None,"[In Kiribati, rough seas, combined with tidal ...",NaN,NaN,NaN,NaN,NaN,NaN,None,MDR55001,KIR,15/05/2015
4,Storm,tropical storm,Papua New Guinea,None,[Port Moresby],"[In Papua New Guinea, two regions experienced ...",NaN,NaN,NaN,NaN,NaN,NaN,Pam,MDR55001,PNG,15/05/2015
